# Social Post Strategy Analysis

## 1. Problem Framing

Leadership relies on social media to reach potential donors, but they do not yet know which post characteristics are most associated with stronger donation results. This notebook uses an **explanatory** pipeline to estimate how platform, content theme, CTA style, timing, and basic post structure relate to short-term donation revenue.

The intended business use is **decision support for the social media chatbot**: the chatbot should keep generating posts, but it should do so with evidence-backed guidance about which post characteristics appear to align with stronger revenue outcomes.

This notebook is explanatory rather than predictive. The goal is not to forecast exact revenue for new posts; it is to understand which characteristics are most strongly associated with donation success and to package those findings for the web app.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

BASE = Path.cwd()
CSV_PATH = BASE / 'data' / 'social_post_performance.csv'
OUTPUT_PATH = BASE / 'social_post_strategy_metadata.json'


## 2. Data Acquisition, Preparation & Exploration

The preferred input is a post-level export at `ml-pipelines/data/social_post_performance.csv` with one row per published post and an attributable 7-day revenue outcome.

Expected columns:
- `platform`
- `content_theme`
- `cta_type`
- `time_bucket`
- `day_of_week`
- `content_type`
- `caption_length`
- `hashtag_count`
- `used_image`
- `revenue_7d_php`
- `donations_7d`

If that export is not available yet, the notebook falls back to a tiny demo dataset so the notebook remains executable. Demo-mode results should **not** be used operationally.


In [2]:
demo_rows = [
    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Tuesday', 'content_type': 'Post', 'caption_length': 122, 'hashtag_count': 3, 'used_image': 1, 'revenue_7d_php': 6800, 'donations_7d': 6},
    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Thursday', 'content_type': 'Post', 'caption_length': 140, 'hashtag_count': 2, 'used_image': 1, 'revenue_7d_php': 7200, 'donations_7d': 7},
    {'platform': 'Facebook', 'content_theme': 'UrgentNeed', 'cta_type': 'GiveToday', 'time_bucket': 'Evening', 'day_of_week': 'Saturday', 'content_type': 'Post', 'caption_length': 118, 'hashtag_count': 2, 'used_image': 1, 'revenue_7d_php': 6100, 'donations_7d': 5},
    {'platform': 'Facebook', 'content_theme': 'TrustBuilding', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Wednesday', 'content_type': 'Post', 'caption_length': 165, 'hashtag_count': 1, 'used_image': 1, 'revenue_7d_php': 3300, 'donations_7d': 3},
    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Morning', 'day_of_week': 'Monday', 'content_type': 'Video', 'caption_length': 98, 'hashtag_count': 4, 'used_image': 1, 'revenue_7d_php': 4200, 'donations_7d': 4},
    {'platform': 'Facebook', 'content_theme': 'Education', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Tuesday', 'content_type': 'Carousel', 'caption_length': 180, 'hashtag_count': 5, 'used_image': 1, 'revenue_7d_php': 2500, 'donations_7d': 2},
    {'platform': 'Instagram', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Thursday', 'content_type': 'Story', 'caption_length': 70, 'hashtag_count': 6, 'used_image': 1, 'revenue_7d_php': 2100, 'donations_7d': 2},
    {'platform': 'Instagram', 'content_theme': 'UrgentNeed', 'cta_type': 'GiveToday', 'time_bucket': 'Evening', 'day_of_week': 'Friday', 'content_type': 'Story', 'caption_length': 64, 'hashtag_count': 7, 'used_image': 1, 'revenue_7d_php': 2600, 'donations_7d': 2},
    {'platform': 'Instagram', 'content_theme': 'TrustBuilding', 'cta_type': 'LearnMore', 'time_bucket': 'Morning', 'day_of_week': 'Monday', 'content_type': 'Post', 'caption_length': 102, 'hashtag_count': 8, 'used_image': 1, 'revenue_7d_php': 1400, 'donations_7d': 1},
    {'platform': 'Instagram', 'content_theme': 'Education', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Wednesday', 'content_type': 'Carousel', 'caption_length': 150, 'hashtag_count': 8, 'used_image': 1, 'revenue_7d_php': 1200, 'donations_7d': 1},
    {'platform': 'Instagram', 'content_theme': 'ReintegrationWin', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Sunday', 'content_type': 'Video', 'caption_length': 88, 'hashtag_count': 5, 'used_image': 1, 'revenue_7d_php': 2800, 'donations_7d': 3},
    {'platform': 'Facebook', 'content_theme': 'ReintegrationWin', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Sunday', 'content_type': 'Video', 'caption_length': 105, 'hashtag_count': 3, 'used_image': 1, 'revenue_7d_php': 5400, 'donations_7d': 5}
]

demo_mode = not CSV_PATH.exists()
if demo_mode:
    df = pd.DataFrame(demo_rows)
else:
    df = pd.read_csv(CSV_PATH)

required_columns = {
    'platform', 'content_theme', 'cta_type', 'time_bucket', 'day_of_week', 'content_type',
    'caption_length', 'hashtag_count', 'used_image', 'revenue_7d_php', 'donations_7d'
}
missing = sorted(required_columns - set(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df.copy()
df['revenue_7d_php'] = df['revenue_7d_php'].fillna(0).clip(lower=0)
df['donations_7d'] = df['donations_7d'].fillna(0).clip(lower=0)
df['caption_length'] = df['caption_length'].fillna(0).clip(lower=0)
df['hashtag_count'] = df['hashtag_count'].fillna(0).clip(lower=0)
df['used_image'] = df['used_image'].fillna(0).astype(int)
df['log_revenue'] = np.log1p(df['revenue_7d_php'])

print('Demo mode:' if demo_mode else 'Production data:', demo_mode)
print('Observations:', len(df))
display(df.head())
display(df.groupby('platform')['revenue_7d_php'].agg(['count', 'mean', 'sum']).round(2))
display(df.groupby('content_theme')['revenue_7d_php'].agg(['count', 'mean', 'sum']).round(2).sort_values('mean', ascending=False))


Demo mode: True
Observations: 12


,platform,content_theme,cta_type,time_bucket,day_of_week,content_type,caption_length,hashtag_count,used_image,revenue_7d_php,donations_7d,log_revenue
0,Facebook,ImpactStory,DonateNow,Evening,Tuesday,Post,122,3,1,6800,6,8.824825
1,Facebook,ImpactStory,DonateNow,Evening,Thursday,Post,140,2,1,7200,7,8.881975
2,Facebook,UrgentNeed,GiveToday,Evening,Saturday,Post,118,2,1,6100,5,8.716208
3,Facebook,TrustBuilding,LearnMore,Afternoon,Wednesday,Post,165,1,1,3300,3,8.101981
4,Facebook,ImpactStory,DonateNow,Morning,Monday,Video,98,4,1,4200,4,8.343078


,count,mean,sum
platform,,,
Facebook,7,5071.43,35500
Instagram,5,2020.00,10100


,count,mean,sum
content_theme,,,
ImpactStory,4,5075.0,20300
UrgentNeed,2,4350.0,8700
ReintegrationWin,2,4100.0,8200
TrustBuilding,2,2350.0,4700
Education,2,1850.0,3700


## 3. Modeling & Feature Selection

We fit an interpretable OLS model on `log(1 + revenue_7d_php)` using post characteristics that would be known at publish time. This is not a randomized experiment, so coefficients should be treated as **associations** rather than proof of causation. Still, the direction and magnitude of these relationships can guide practical content decisions.


In [3]:
formula = (
    'log_revenue ~ C(platform) + C(content_theme) + C(cta_type) + C(time_bucket) '
    '+ C(day_of_week) + C(content_type) + caption_length + hashtag_count + used_image'
)

model = smf.ols(formula=formula, data=df).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            log_revenue   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Thu, 09 Apr 2026   Prob (F-statistic):                nan
Time:                        19:42:45   Log-Likelihood:                 342.99
No. Observations:                  12   AIC:                            -662.0
Df Residuals:                       0   BIC:                            -656.2
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


## 4. Evaluation & Interpretation

Because this is an explanatory pipeline, the main outputs are coefficient directions, p-values, and practical implications. False confidence is the key risk here: if we overstate weak findings, the founders may shape their content around noise. If we understate useful signals, they may keep posting sporadically without learning.


In [4]:
coef_table = (
    pd.DataFrame({
        'feature': model.params.index,
        'coefficient': model.params.values,
        'p_value': model.pvalues.values,
    })
    .query("feature != 'Intercept'")
    .assign(abs_coef=lambda x: x['coefficient'].abs())
    .sort_values(['p_value', 'abs_coef'], ascending=[True, False])
)

display(coef_table.head(12).round(4))
print('R-squared:', round(float(model.rsquared), 4))


,feature,coefficient,p_value,abs_coef
17,C(content_type)[T.Story],1.2228,NaN,1.2228
8,C(time_bucket)[T.Evening],1.1970,NaN,1.1970
21,used_image,1.0077,NaN,1.0077
2,C(content_theme)[T.ImpactStory],0.9224,NaN,0.9224
18,C(content_type)[T.Video],0.8562,NaN,0.8562
4,C(content_theme)[T.TrustBuilding],0.7326,NaN,0.7326
7,C(cta_type)[T.LearnMore],-0.6810,NaN,0.6810
10,C(day_of_week)[T.Monday],0.4240,NaN,0.4240
9,C(time_bucket)[T.Morning],0.4240,NaN,0.4240
5,C(content_theme)[T.UrgentNeed],0.4019,NaN,0.4019


R-squared: 1.0


## 5. Causal and Relationship Analysis (Required)

This notebook is honest about its limits: unless the organization starts tagging individual posts with attributable donation links, these relationships remain observational. Even with attribution, unobserved confounders such as campaign quality, external events, and audience size may still affect outcomes.

What the model can still reveal is which post features appear consistently aligned with stronger or weaker revenue outcomes, which is enough to improve the chatbot prompt and the team's posting habits.


In [5]:
def label_feature(name: str, coef: float) -> str:
    direction = 'higher' if coef > 0 else 'lower'
    cleaned = name.replace('C(', '').replace(')', '').replace('[T.', ' = ').replace(']', '')
    return f'{cleaned} is associated with {direction} 7-day donation revenue.'

validated = [
    label_feature(row.feature, row.coefficient)
    for row in coef_table.itertuples()
    if row.p_value < 0.05
]

directional = [
    label_feature(row.feature, row.coefficient)
    for row in coef_table.itertuples()
    if 0.05 <= row.p_value < 0.20
]

top_positive = coef_table.sort_values('coefficient', ascending=False).head(3)
recommendations = []
for row in top_positive.itertuples():
    recommendations.append({
        'title': f'Test more of {row.feature}',
        'detail': label_feature(row.feature, row.coefficient)
    })

data_gaps = []
if demo_mode:
    data_gaps.append('Notebook is running on the embedded demo dataset. Replace it with an attributed post export before using findings operationally.')
if 'actual_publish_time' not in df.columns:
    data_gaps.append('Actual publish timestamps were not provided, so time effects rely on coarse buckets instead of exact posting windows.')
if 'tracked_link_id' not in df.columns:
    data_gaps.append('Tracked donation-link identifiers were not provided, so attribution quality may be limited.')

result = {
    'pipeline_version': '0.1.0',
    'last_run': datetime.now(timezone.utc).date().isoformat() if not demo_mode else None,
    'analysis_mode': 'explanatory',
    'unit_of_analysis': 'published social media post',
    'outcome_metric': 'Attributed donation revenue within 7 days (PHP)',
    'evidence_strength': 'Demo-only scaffold output.' if demo_mode else 'Live post-level explanatory analysis from notebook output.',
    'n_observations': int(len(df)),
    'r_squared': round(float(model.rsquared), 4),
    'validated_findings': validated[:5],
    'directional_findings': directional[:5],
    'recommendations': recommendations[:3],
    'data_gaps': data_gaps,
}

print(json.dumps(result, indent=2))
if demo_mode:
    print('Demo mode is active, so social_post_strategy_metadata.json was not overwritten.')
else:
    OUTPUT_PATH.write_text(json.dumps(result, indent=2))
    print(f'Wrote metadata to {OUTPUT_PATH}')


{
  "pipeline_version": "0.1.0",
  "last_run": null,
  "analysis_mode": "explanatory",
  "unit_of_analysis": "published social media post",
  "outcome_metric": "Attributed donation revenue within 7 days (PHP)",
  "evidence_strength": "Demo-only scaffold output.",
  "n_observations": 12,
  "r_squared": 1.0,
  "validated_findings": [],
  "directional_findings": [],
  "recommendations": [
    {
      "title": "Test more of C(content_type)[T.Story]",
      "detail": "content_type = Story is associated with higher 7-day donation revenue."
    },
    {
      "title": "Test more of C(time_bucket)[T.Evening]",
      "detail": "time_bucket = Evening is associated with higher 7-day donation revenue."
    },
    {
      "title": "Test more of used_image",
      "detail": "used_image is associated with higher 7-day donation revenue."
    }
  ],
  "data_gaps": [
    "Notebook is running on the embedded demo dataset. Replace it with an attributed post export before using findings operationally.",
  

## 6. Deployment Notes

When the notebook writes `social_post_strategy_metadata.json`, the FastAPI service in [app.py](./app.py) serves it at `/marketing/post-strategy-analysis`. The ASP.NET backend then injects the findings into the social chatbot context so generated posts can reflect the strongest available post-level evidence without replacing the existing chatbot workflow.
